# NB23 게임이론 — 23장

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB23-1: 이기적 채굴 몬테카를로 — 이얄–시러의 상태기계(선두 0, 0′, 1, 2, …)를 사건 단위로 돌려 풀의 몫 R_pool 을 잰다 (공부용)
import random

def selfish_mining_share(alpha: float, gamma: float, n_events: int, seed: int = 23) -> float:
    """블록 발견 사건을 n_events 번 뽑는다(풀이 찾을 확률 α). 돌려주는 값: 주 체인에 남은 블록 중 풀의 비율."""
    rng = random.Random(seed)
    lead = 0                 # 풀의 비공개 선두(블록 수)
    tie = False              # 상태 0′: 길이 1인 두 공개 가지가 겨루는 중
    pool_blocks = others_blocks = 0
    for _ in range(n_events):
        pool_found = rng.random() < alpha
        if tie:                                   # 상태 0′ — 세 갈래 모두 상태 0 으로
            if pool_found:                        # (b) 풀이 자기 가지 위에 찾음 → 풀 2
                pool_blocks += 2
            elif rng.random() < gamma:            # (c) 나머지가 풀의 가지 위에 찾음 → 풀 1, 나머지 1
                pool_blocks += 1; others_blocks += 1
            else:                                 # (d) 나머지가 자기 가지 위에 찾음 → 나머지 2
                others_blocks += 2
            tie = False; lead = 0
            continue
        if pool_found:                            # (a) 풀이 찾음 → 선두 +1 (보상은 나중에)
            lead += 1
            continue
        # 나머지가 찾음
        if lead == 0:                             # (e) 비공개 가지 없음 → 나머지 1
            others_blocks += 1
        elif lead == 1:                           # (f) 선두 1 → 풀이 숨긴 블록을 공개, 겨루기 시작
            tie = True; lead = 0
        elif lead == 2:                           # (g) 선두 2 → 풀이 두 블록 공개, 나머지 블록은 버려짐 → 풀 2
            pool_blocks += 2; lead = 0
        else:                                     # (h) 선두 > 2 → 풀이 한 블록 공개(풀 1), 나머지 블록은 결국 버려짐
            pool_blocks += 1; lead -= 1
    return pool_blocks / (pool_blocks + others_blocks)

def R_pool_closed_form(a: float, g: float) -> float:
    """CACM판 식 (3) = arXiv v5 식 (8): R_pool = [α(1−α)²(4α+γ(1−2α)) − α³] / [1 − α(1+(2−α)α)], 0 ≤ α ≤ 1/2"""
    return (a * (1 - a) ** 2 * (4 * a + g * (1 - 2 * a)) - a ** 3) / (1 - a * (1 + (2 - a) * a))

N = 400_000
print(f"사건 {N:,}번. 정직한 채굴의 몫은 α 그 자체. 아래는 이기적 채굴의 몫(시뮬레이션 / 닫힌 식).")
print(f"{'α':>5} {'γ':>5} {'시뮬 R_pool':>12} {'식 R_pool':>10} {'정직 α':>8} {'이기적이 유리?':>12}")
for g in [0.0, 0.5, 1.0]:
    for a in [0.10, 0.20, 0.25, 0.30, 0.34, 0.40, 0.45]:
        sim = selfish_mining_share(a, g, N)
        cf = R_pool_closed_form(a, g)
        print(f"{a:>5.2f} {g:>5.1f} {sim:>12.4f} {cf:>10.4f} {a:>8.2f} {'예' if cf > a else '아니오':>12}")

사건 400,000번. 정직한 채굴의 몫은 α 그 자체. 아래는 이기적 채굴의 몫(시뮬레이션 / 닫힌 식).
    α     γ    시뮬 R_pool   식 R_pool     정직 α     이기적이 유리?
 0.10   0.0       0.0358     0.0356     0.10          아니오
 0.20   0.0       0.1300     0.1297     0.20          아니오
 0.25   0.0       0.1948     0.1951     0.25          아니오
 0.30   0.0       0.2713     0.2731     0.30          아니오
 0.34   0.0       0.3446     0.3463     0.34            예
 0.40   0.0       0.4813     0.4837     0.40            예
 0.45   0.0       0.6488     0.6518     0.45            예
 0.10   0.5       0.0724     0.0724     0.10          아니오
 0.20   0.5       0.1830     0.1824     0.20          아니오
 0.25   0.5       0.2501     0.2500     0.25          아니오
 0.30   0.5       0.3254     0.3269     0.30            예
 0.34   0.5       0.3953     0.3970     0.34            예
 0.40   0.5       0.5235     0.5256     0.40            예
 0.45   0.5       0.6778     0.6806     0.45            예
 0.10   1.0       0.1097     0.1092     0.10            예
 0.20   1.0

In [ ]:
# NB23-2: 2×2 이득표와 「문턱」 — 작은 참여자 둘의 공개/보류 게임을 이기적 채굴의 몫으로 채워 넣어 균형을 읽는다 (공부용)
# 두 풀 P(점유율 a), Q(점유율 b)가 각각 「정직(즉시 공개)」 또는 「보류(이기적 채굴)」를 고른다. 나머지(1−a−b)는 정직.
# 단순화: 보류하는 풀이 하나뿐이면 그 몫은 R_pool(α, γ); 둘 다 보류하면 서로 상대의 숨긴 블록을 모르므로 각자 R_pool 을 쓰되 상대도 보류하는 효과는 무시한다(한계 명시).
def payoff(a, b, P_selfish, Q_selfish, gamma):
    rp = R_pool_closed_form(a, gamma) if P_selfish else a
    rq = R_pool_closed_form(b, gamma) if Q_selfish else b
    return rp, rq

for gamma in [0.0, 0.5]:
    print(f"\nγ = {gamma}: 문턱 (1−γ)/(3−2γ) = {(1-gamma)/(3-2*gamma):.3f}")
    for a, b in [(0.20, 0.20), (0.30, 0.30), (0.40, 0.20)]:
        print(f"  P={a:.2f}, Q={b:.2f}  (행: P의 선택, 열: Q의 선택; 칸: P의 몫, Q의 몫)")
        print(f"  {'':>10}{'Q 정직':>18}{'Q 보류':>18}")
        for ps, pl in [(False, "P 정직"), (True, "P 보류")]:
            cells = []
            for qs in [False, True]:
                rp, rq = payoff(a, b, ps, qs, gamma)
                cells.append(f"({rp:.3f}, {rq:.3f})")
            print(f"  {pl:>10}{cells[0]:>18}{cells[1]:>18}")
        best_P = "보류" if R_pool_closed_form(a, gamma) > a else "정직"
        best_Q = "보류" if R_pool_closed_form(b, gamma) > b else "정직"
        print(f"  → P의 우월전략 {best_P}, Q의 우월전략 {best_Q}  (점유율이 문턱 아래면 정직이, 위면 보류가 우월하다)")
print("\n이 표는 「상대가 무엇을 하든 내 몫이 문턱으로 정해진다」는 단순화 위에 있다. 두 풀이 동시에 보류하는 경우의 상호작용은 모형 밖이다.")


γ = 0.0: 문턱 (1−γ)/(3−2γ) = 0.333
  P=0.20, Q=0.20  (행: P의 선택, 열: Q의 선택; 칸: P의 몫, Q의 몫)
                          Q 정직              Q 보류
        P 정직    (0.200, 0.200)    (0.200, 0.130)
        P 보류    (0.130, 0.200)    (0.130, 0.130)
  → P의 우월전략 정직, Q의 우월전략 정직  (점유율이 문턱 아래면 정직이, 위면 보류가 우월하다)
  P=0.30, Q=0.30  (행: P의 선택, 열: Q의 선택; 칸: P의 몫, Q의 몫)
                          Q 정직              Q 보류
        P 정직    (0.300, 0.300)    (0.300, 0.273)
        P 보류    (0.273, 0.300)    (0.273, 0.273)
  → P의 우월전략 정직, Q의 우월전략 정직  (점유율이 문턱 아래면 정직이, 위면 보류가 우월하다)
  P=0.40, Q=0.20  (행: P의 선택, 열: Q의 선택; 칸: P의 몫, Q의 몫)
                          Q 정직              Q 보류
        P 정직    (0.400, 0.200)    (0.400, 0.130)
        P 보류    (0.484, 0.200)    (0.484, 0.130)
  → P의 우월전략 보류, Q의 우월전략 정직  (점유율이 문턱 아래면 정직이, 위면 보류가 우월하다)

γ = 0.5: 문턱 (1−γ)/(3−2γ) = 0.250
  P=0.20, Q=0.20  (행: P의 선택, 열: Q의 선택; 칸: P의 몫, Q의 몫)
                          Q 정직              Q 보류
        P 정직    (0.200, 0.200)    (0.200, 0.182)
